# **Training Pipeline**

Quy trình huấn luyện seq2seq cho tóm tắt văn bản tiếng Việt

### Lưu ý quan trọng khi dùng TPU:
- Dùng **bf16** (TPU v5e hỗ trợ native bfloat16)
- **Không dùng** CUDA quantization (BitsAndBytes không hỗ trợ TPU)
- Cần cài **torch_xla** để PyTorch giao tiếp với TPU
- HuggingFace Trainer tự động detect TPU khi có torch_xla

In [ ]:
from __future__ import annotations
import os

# Giảm phân mảnh CUDA allocator; phải đặt trước khi model/GPU được khởi tạo.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


In [5]:
from __future__ import annotations
import os

REPO_URL = "https://github.com/dungcony/sumarization.git"

# Khắc phục lỗi clone lồng nhau (lỗi 2 thư mục sumarization)
if os.path.exists(".git") and "sumarization" in os.getcwd():
    print("Đang cập nhật code mới nhất...")
    !git pull
else:
    print("Đang tải mã nguồn...")
    !git clone {REPO_URL} sumarization
    %cd sumarization

Đang cập nhật code mới nhất...


302.98s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Already up to date.


In [6]:
%pip install -e . 2>&1 | tail -5

311.80s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Attempting uninstall: vn-summarization
    Found existing installation: vn-summarization 1.0.0
    Uninstalling vn-summarization-1.0.0:
      Successfully uninstalled vn-summarization-1.0.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import warnings
import threading

warnings.filterwarnings('ignore')  # Ẩn các cảnh báo phiền phức

import time
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

In [8]:
from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import (
    format_duration,
    format_number,
    save_json,
    set_seed,
    setup_logger,
    count_parameters,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

✅ Import thành công!


In [18]:
CONFIG_FILE = "configs/vit5_base_phase_1.yaml"
config = load_config(CONFIG_FILE)

colab_output_model = Path("/content") / config.training.output_dir
kaggle_output_model = Path("/kaggle/working") / config.training.output_dir

is_kaggle = Path("/kaggle").exists()

output_model = (
    kaggle_output_model
    if is_kaggle
    else colab_output_model
)

best_dir = output_model / "best"

# Cập nhật lại config để Trainer lưu đúng đường dẫn tuyệt đối
config = apply_overrides(config, {
    "training.output_dir": str(output_model),
})

print(f"Colab output model: {colab_output_model}")
print(f"Kaggle output model: {kaggle_output_model}")
print(f"Output đang sử dụng: {output_model}")
print(f"Best checkpoint: {best_dir}")
print(config.data)

Colab output model: /content/outputs_phase_1/vit5_base
Kaggle output model: /kaggle/working/outputs_phase_1/vit5_base
Output đang sử dụng: /content/outputs_phase_1/vit5_base
Best checkpoint: /content/outputs_phase_1/vit5_base/best
DataConfig(train_file='data/phase_1/train_*.**', valid_file='data/phase_1/validation_*.**', test_file='data/phase_1/test_*.**', source_prefix='summarize: ', max_source_length=768, max_target_length=256, max_train_samples=None, max_eval_samples=None)


---
## 2. Kiểm tra TPU & Cài đặt

In [14]:
# Kiểm tra TPU bằng torch_xla; Kaggle TPU có thể không đặt TPU_NAME.
tpu_name = os.environ.get('TPU_NAME', os.environ.get('COLAB_TPU_ADDR', None))
is_tpu = False
print(f"TPU_NAME: {tpu_name}")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm

    device = xm.xla_device()
    is_tpu = str(device).startswith("xla")
    print(f"✅ TPU sẵn sàng! Device: {device}")
    print(f"   torch_xla version: {torch_xla.__version__}")
except Exception as e:
    print(f"ℹ️ Không dùng TPU: {e}")

print(f"Runtime đang dùng: {'TPU' if is_tpu else 'GPU/CPU'}")

TPU_NAME: None
❌ Không tìm thấy TPU: No module named 'torch_xla'
Hãy chắc chắn đã chọn Runtime > Change runtime type > TPU v5e


---
## 3. Cấu hình phần cứng  (đọc phần cứng để áp dụng gput hoặc tpu)

In [16]:
import torch

# Giữ kết quả phát hiện thực tế bằng torch_xla từ cell trước.
is_tpu = bool(globals().get("is_tpu", False))
is_gpu = torch.cuda.is_available() and not is_tpu
hardware = "TPU" if is_tpu else "GPU" if is_gpu else "CPU"

# Ghi đè các thiết lập cho Lần 1
config = apply_overrides(config, {
    "training.num_train_epochs": 4,
    # Precision & Optimizer
    "training.precision": "fp16",
    # Adafactor tiết kiệm đáng kể optimizer state cho T5 trên GPU 16 GB.
    "training.optim": "adafactor",

    # Thư mục lưu kết quả Lần 1
    "training.output_dir": output_model,

    # Batch size
    "training.per_device_train_batch_size": 8,
    "training.per_device_eval_batch_size": 1,
    "training.gradient_accumulation_steps": 2,

    "training.gradient_checkpointing": False,
    # LabelSmoother tạo tensor log_softmax rất lớn; log OOM xảy ra đúng tại đây.
    "training.label_smoothing_factor": 0.0 if is_gpu else config.training.label_smoothing_factor,

    # --- CỤM CHẠY THỬ ---
    # "data.max_train_samples": 200,   
    # "data.max_eval_samples": 50,     
    # "training.max_steps": 20,
    "training.logging_steps": 10,

})

print(f"🔹 TRAIN LẦN {config.phase.name}: Học nền tảng (Parquet tổng hợp)")
print(f"Hardware:   {hardware}")
print(f"Model:      {config.model.name_or_path}")
print(f"Train data: {config.data.train_file}")
print(f"Precision:  {config.training.precision}")
print(f"Batch size: {config.training.per_device_train_batch_size}")
print(f"Eval batch: {config.training.per_device_eval_batch_size}")
print(f"LR:         {config.training.learning_rate}")
print(f"Optimizer:  {config.training.optim}")
print(f"Grad ckpt:  {config.training.gradient_checkpointing}")
print(f"Label smooth: {config.training.label_smoothing_factor}")
print(f"Output:     {config.training.output_dir}")


🔹 TRAIN LẦN phase_2: Học nền tảng (Parquet tổng hợp)
Hardware:   GPU
Model:      /kaggle/working/outputs_phase_1/vit5_base/best
Train data: data/phase_2/train_*.*
Precision:  auto
Batch size: 4
LR:         5e-06
Optimizer:  adamw_torch
Output:     /content/outputs_phase_2/vit5_base


---
## 4. Tải Model & Data

In [19]:
# Thiết lập seed
set_seed(config.training.seed)

# Tải tokenizer
tokenizer = load_tokenizer(config.model)
print(f"✅ Tokenizer: vocab_size={tokenizer.vocab_size}")

[INFO] src.model: Đang tải T5 SentencePiece tokenizer cho: VietAI/vit5-base
[INFO] src.model: Đã tải tokenizer: vocab_size=36000, type=T5Tokenizer


✅ Tokenizer: vocab_size=36000


In [20]:
# Tải model
model = load_model(config.model, tokenizer, config.generation)

# Áp dụng gradient checkpointing (tiết kiệm bộ nhớ TPU)
if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)

# Đóng băng encoder nếu cần
if config.training.freeze_encoder:
    freeze_encoder(model)

# Áp dụng LoRA nếu bật
model = apply_lora(model, config.lora)

params = count_parameters(model)
print(f"✅ Model loaded: {format_number(params['total'])} total, "
      f"{format_number(params['trainable'])} trainable ({params['trainable_percent']}%)")

[INFO] src.model: Đang tải mô hình: VietAI/vit5-base
[INFO] src.model: Đã tải mô hình: 225,950,976 tổng số tham số, 225,950,976 có thể huấn luyện (100.0%)
[INFO] src.model: LoRA bị vô hiệu hóa, sử dụng fine-tuning toàn bộ (full fine-tuning)


✅ Model loaded: 225,950,976 total, 225,950,976 trainable (100.0%)


In [21]:
# Tải & tiền xử lý dữ liệu
datasets = load_and_preprocess(tokenizer, config.data)

print(f"✅ Train:      {len(datasets['train'])} samples")
print(f"✅ Validation: {len(datasets['validation'])} samples")
if 'test' in datasets:
    print(f"✅ Test:       {len(datasets['test'])} samples")


[INFO] src.data: Đang tải dữ liệu: train=1 file; validation=1 file; test=1 file
[INFO] src.data: Đã tải tập dữ liệu: 10775 train, 1348 validation, 1344 test
[INFO] src.data: train: đã tokenize 10775 mẫu
[INFO] src.data: validation: đã tokenize 1348 mẫu
[INFO] src.data: test: đã tokenize 1344 mẫu


✅ Train:      10775 samples
✅ Validation: 1348 samples
✅ Test:       1344 samples


---
## 5. Cấu hình Trainer

In [22]:
tc = config.training
output_dir = Path(tc.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Xác định precision
fp16 = tc.precision == "fp16"
bf16 = tc.precision == "bf16"

training_args = Seq2SeqTrainingArguments(
    output_dir=tc.output_dir,
    seed=tc.seed,

    # Epochs & steps
    num_train_epochs=tc.num_train_epochs,
    max_steps=tc.max_steps,

    # Batch size
    per_device_train_batch_size=tc.per_device_train_batch_size,
    per_device_eval_batch_size=tc.per_device_eval_batch_size,
    gradient_accumulation_steps=tc.gradient_accumulation_steps,

    # Optimizer
    learning_rate=tc.learning_rate,
    weight_decay=tc.weight_decay,
    warmup_ratio=tc.warmup_ratio,
    lr_scheduler_type=tc.lr_scheduler_type,
    optim=tc.optim,  #thuật toán cập nhật kiến thức

    # Regularization
    label_smoothing_factor=tc.label_smoothing_factor,

    # Precision — TPU v5e dùng bf16
    fp16=fp16,
    bf16=bf16,

    # Evaluation & saving
    eval_strategy=tc.eval_strategy,  #số bước học cho mỗi lần kiểm tra
    eval_steps=tc.eval_steps,
    save_strategy=tc.save_strategy,  # số bước học cho mỗi lần lưu
    save_steps=tc.save_steps,
    save_total_limit=tc.save_total_limit,  # số bản lưu trữ được giữ lại
    logging_strategy="steps",
    logging_steps=tc.logging_steps,
    logging_first_step=True,
    # Tắt progress bar dạng ghi đè; callback bên dưới sẽ in log thật ra stdout Kaggle.
    disable_tqdm=True,

    # Best model
    metric_for_best_model=tc.metric_for_best_model,
    greater_is_better=tc.greater_is_better,
    load_best_model_at_end=tc.load_best_model_at_end,  #chọn bản tốt nhất để lưu

    # Generation during eval
    predict_with_generate=True,  #yêu cầu tạo ra 1 bài báo và tóm tắt rồi mới chấm điểm
    generation_max_length=config.generation.max_length,  #số token tối đa được viết khi làm bài kiểm tra

    # Logging — dùng tensorboard
    report_to=["tensorboard"],
    logging_dir=str(output_dir / "logs"),

    # ===== TPU-specific =====
    # dataloader_drop_last giúp tránh batch cuối bị lẻ trên TPU
    dataloader_drop_last=True,
)

print(f"✅ TrainingArguments đã sẵn sàng")
print(f"   bf16={bf16}, fp16={fp16}")
print(f"   device: {training_args.device}")

PermissionError: [Errno 13] Permission denied: '/content'

In [ ]:
class KaggleProgressCallback(TrainerCallback):
    """In tiến trình thật ra stdout và file, kể cả khi một training step đang chạy lâu."""

    def __init__(self, label, log_every_steps=10, heartbeat_seconds=60, log_file=None):
        self.label = label
        self.log_every_steps = max(1, int(log_every_steps))
        self.heartbeat_seconds = max(10, int(heartbeat_seconds))
        self.log_file = Path(log_file) if log_file else None
        self.started_at = None
        self.last_completed_step = 0
        self.max_steps = 0
        self._is_main_process = True
        self._stop_event = threading.Event()
        self._write_lock = threading.Lock()
        self._heartbeat_thread = None

    def _device_status(self):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.memory_reserved() / (1024 ** 3)
            return f"GPU RAM={allocated:.2f}/{reserved:.2f} GB (allocated/reserved)"
        if globals().get("is_tpu", False):
            return "TPU/XLA đang hoạt động"
        return "CPU đang hoạt động"

    def _emit(self, message):
        if not self._is_main_process:
            return
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] [{self.label}] {message}"
        with self._write_lock:
            print(line, flush=True)
            if self.log_file is not None:
                self.log_file.parent.mkdir(parents=True, exist_ok=True)
                with self.log_file.open("a", encoding="utf-8") as handle:
                    handle.write(line + "\n")

    def _progress_text(self, step=None):
        step = self.last_completed_step if step is None else step
        if not self.started_at:
            return f"step={step}/{self.max_steps}"
        elapsed = time.time() - self.started_at
        percent = 100 * step / self.max_steps if self.max_steps else 0
        if step > 0 and self.max_steps > step:
            eta = elapsed / step * (self.max_steps - step)
            eta_text = format_duration(eta)
        else:
            eta_text = "đang tính"
        return (
            f"step={step}/{self.max_steps} ({percent:.1f}%) | "
            f"đã chạy={format_duration(elapsed)} | ETA={eta_text}"
        )

    def _heartbeat_loop(self):
        while not self._stop_event.wait(self.heartbeat_seconds):
            self._emit(
                "♥ VẪN ĐANG TRAIN | "
                f"{self._progress_text()} | {self._device_status()}"
            )

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        self.last_completed_step = state.global_step
        self.max_steps = state.max_steps
        self._is_main_process = state.is_world_process_zero
        self._stop_event.clear()
        self._emit(
            f"TRAINER ĐÃ VÀO VÒNG LẶP | {self._progress_text()} | "
            f"log mỗi {self.log_every_steps} step, heartbeat mỗi {self.heartbeat_seconds}s"
        )
        if self._is_main_process:
            self._heartbeat_thread = threading.Thread(
                target=self._heartbeat_loop,
                name=f"{self.label}-heartbeat",
                daemon=True,
            )
            self._heartbeat_thread.start()

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch_number = int(state.epoch or 0) + 1
        self._emit(f"Bắt đầu epoch {epoch_number}/{int(args.num_train_epochs)}")

    def on_step_begin(self, args, state, control, **kwargs):
        next_step = state.global_step + 1
        if next_step == 1 or next_step % self.log_every_steps == 0:
            self._emit(f"Đang xử lý optimizer step {next_step}/{state.max_steps}...")

    def on_step_end(self, args, state, control, **kwargs):
        self.last_completed_step = state.global_step
        if state.global_step == 1 or state.global_step % self.log_every_steps == 0:
            self._emit(f"Đã hoàn tất | {self._progress_text()} | {self._device_status()}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        fields = []
        for key in ("loss", "learning_rate", "grad_norm", "epoch"):
            if key in logs:
                value = logs[key]
                fields.append(f"{key}={value:.6g}" if isinstance(value, (int, float)) else f"{key}={value}")
        if fields:
            self._emit("METRICS | " + " | ".join(fields))

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if key in ("eval_loss", "eval_rouge1", "eval_rouge2", "eval_rougeL")
        )
        self._emit(f"ĐÁNH GIÁ XONG | {summary or 'không có metric tóm tắt'}")

    def on_save(self, args, state, control, **kwargs):
        self._emit(f"ĐÃ LƯU CHECKPOINT tại step {state.global_step}")

    def stop(self):
        self._stop_event.set()
        if self._heartbeat_thread and self._heartbeat_thread.is_alive():
            self._heartbeat_thread.join(timeout=2)

    def on_train_end(self, args, state, control, **kwargs):
        self.last_completed_step = state.global_step
        self._emit(f"KẾT THÚC TRAIN | {self._progress_text()}")
        self.stop()


# Data collator (dynamic padding)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

# Hàm tính ROUGE metrics
compute_metrics = build_compute_metrics(tokenizer)

# Callback tiến trình: vừa hiện trong Kaggle Logs, vừa lưu vào file để kiểm tra lại.
progress_callback = KaggleProgressCallback(
    label="PHASE 1",
    log_every_steps=tc.logging_steps,
    heartbeat_seconds=60,
    log_file=output_dir / "training_progress.log",
)
callbacks = [progress_callback]
if tc.early_stopping_patience > 0:
    callbacks.append(
        EarlyStoppingCallback(
            early_stopping_patience=tc.early_stopping_patience,
        )
    )
    print(f"✅ Early stopping: patience={tc.early_stopping_patience}")

# Xây dựng Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print("✅ Trainer đã sẵn sàng!")

---
## 6. Huấn luyện 🚀

In [ ]:
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN")
print("=" * 60)
print(f"  Model:       {config.model.name_or_path}")
print(f"  Epochs:      {tc.num_train_epochs}")
print(f"  Batch size:  {tc.per_device_train_batch_size}")
print(f"  Grad accum:  {tc.gradient_accumulation_steps}")
print(f"  GPU count:   {training_args.n_gpu}")
print(
    f"  Effective batch: "
    f"{tc.per_device_train_batch_size * max(1, training_args.n_gpu) * tc.gradient_accumulation_steps}"
)
print(f"  LR:          {tc.learning_rate}")
print(f"  Optimizer:   {tc.optim}")
print(f"  Precision:   {tc.precision}")
print(f"  LoRA:        {config.lora.enabled}")
print(f"  Train samples: {len(datasets['train'])}")
print(f"  Log file:    {output_dir / 'training_progress.log'}")
print("  Dấu hiệu đang chạy: dòng '♥ VẪN ĐANG TRAIN' sẽ xuất hiện mỗi 60 giây")
print("=" * 60)

if is_gpu:
    torch.cuda.empty_cache()

start_time = time.time()

try:
    train_result = trainer.train(
        resume_from_checkpoint=tc.resume_from_checkpoint,
    )
finally:
    # Dừng heartbeat cả khi người dùng interrupt hoặc training phát sinh lỗi.
    progress_callback.stop()

elapsed = time.time() - start_time
print(f"\n✅ Huấn luyện hoàn thành! Thời gian: {format_duration(elapsed)}")

---
## 7. Lưu model & Đánh giá

In [ ]:
# Lưu model tốt nhất

print(f"Đang lưu model tốt nhất tới: {best_dir}")
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
print("✅ Đã lưu model!")

In [ ]:
# Đánh giá cuối cùng trên tập VALIDATION
print("Đang chạy đánh giá trên tập Validation...")
eval_results = trainer.evaluate(metric_key_prefix="eval")

print("\n" + "=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ - TẬP VALIDATION")
print("=" * 60)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print("=" * 60)

# Đánh giá trên tập TEST (Bài thi Đại học - Độc lập hoàn toàn)
if 'test' in datasets:
    print("\nĐang đánh giá trên tập TEST...")
    test_results = trainer.evaluate(eval_dataset=datasets['test'], metric_key_prefix='test')
    print("\n" + "=" * 60)
    print("KẾT QUẢ ĐÁNH GIÁ - TẬP TEST (Khách quan nhất)")
    print("=" * 60)
    for k, v in sorted(test_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
    print("=" * 60)
else:
    print("\n⚠️ Không tìm thấy tập TEST trong dữ liệu.")
    test_results = {}


---
## 8. So sánh với model gốc

Sau khi model fine-tuned đã được lưu và đánh giá, nạp lại checkpoint gốc `VietAI/vit5-base`, chấm trên đúng tập test và cùng cấu hình generation.

In [ ]:
import gc

if "test" not in datasets:
    raise ValueError("Cần tập test độc lập để so sánh model gốc và model fine-tuned.")

# Kết quả model fine-tuned đã nằm trong test_results; giải phóng model khỏi RAM/TPU
# trước khi nạp lại checkpoint gốc để tránh giữ đồng thời hai model.
del trainer
del model
del data_collator
gc.collect()
if is_tpu and "xm" in globals():
    xm.mark_step()

print("=" * 72)
print("ĐÁNH GIÁ MODEL GỐC SAU KHI ĐÃ HOÀN TẤT FINE-TUNE")
print("=" * 72)
print(f"Model gốc: {config.model.name_or_path}")
print(f"Model fine-tuned đã lưu: {best_dir}")
print(f"Số mẫu validation dùng chung: {len(datasets['validation'])}")
print(f"Số mẫu test dùng chung: {len(datasets['test'])}")

# Nạp mới từ Hugging Face ID trong config; đây là trọng số gốc, không phải best_dir.
baseline_model = load_model(config.model, tokenizer, config.generation)
baseline_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=baseline_model,
    padding=True,
    label_pad_token_id=-100,
)
baseline_trainer = Seq2SeqTrainer(
    model=baseline_model,
    args=training_args,
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=baseline_data_collator,
    compute_metrics=compute_metrics,
)

baseline_validation_results = baseline_trainer.evaluate(
    eval_dataset=datasets["validation"],
    metric_key_prefix="baseline_eval",
)
baseline_test_results = baseline_trainer.evaluate(
    eval_dataset=datasets["test"],
    metric_key_prefix="baseline_test",
)

print("\nKẾT QUẢ MODEL GỐC TRÊN TẬP VALIDATION")
for key, value in sorted(baseline_validation_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print("\nKẾT QUẢ MODEL GỐC TRÊN TẬP TEST")
for key, value in sorted(baseline_test_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

save_json(baseline_validation_results, output_dir / "baseline_validation_results.json")
save_json(baseline_test_results, output_dir / "baseline_test_results.json")
print(f"✅ Đã lưu baseline validation/test tại: {output_dir}")

del baseline_trainer
del baseline_model
del baseline_data_collator
gc.collect()

### So sánh trực tiếp trước và sau fine-tune

In [ ]:
comparison_results = {
    "baseline_model": config.model.name_or_path,
    "finetuned_model": str(best_dir),
    "selection_metric": "validation_rougeL",
    "test_samples": len(datasets["test"]),
    "metrics": {},
}

# Dùng validation để quyết định có chạy phase 2 hay không; không dùng test để chọn model.
baseline_validation_rouge_l = baseline_validation_results["baseline_eval_rougeL"]
finetuned_validation_rouge_l = eval_results["eval_rougeL"]
validation_rouge_l_delta = finetuned_validation_rouge_l - baseline_validation_rouge_l
phase_1_better_than_baseline = validation_rouge_l_delta > 0
comparison_results["phase_2_gate"] = {
    "baseline_validation_rougeL": baseline_validation_rouge_l,
    "finetuned_validation_rougeL": finetuned_validation_rouge_l,
    "absolute_improvement": validation_rouge_l_delta,
    "phase_1_better_than_baseline": phase_1_better_than_baseline,
}

print("\nĐIỀU KIỆN CHẠY PHASE 2 — VALIDATION ROUGE-L")
print(f"  Model gốc:       {baseline_validation_rouge_l:.2f}")
print(f"  Sau phase 1:     {finetuned_validation_rouge_l:.2f}")
print(f"  Chênh lệch:      {validation_rouge_l_delta:+.2f}")
print(f"  Chạy phase 2:    {phase_1_better_than_baseline}")

print("\n" + "=" * 72)
print("SO SÁNH MODEL GỐC VÀ MODEL SAU FINE-TUNE TRÊN CÙNG TẬP TEST")
print("=" * 72)
print(f"{'Metric':<12} {'Trước FT':>12} {'Sau FT':>12} {'Chênh lệch':>14}")
print("-" * 72)

for metric in ("rouge1", "rouge2", "rougeL"):
    baseline_value = baseline_test_results[f"baseline_test_{metric}"]
    finetuned_value = test_results[f"test_{metric}"]
    delta = finetuned_value - baseline_value
    comparison_results["metrics"][metric] = {
        "before_finetune": baseline_value,
        "after_finetune": finetuned_value,
        "absolute_improvement": delta,
    }
    print(f"{metric:<12} {baseline_value:>12.2f} {finetuned_value:>12.2f} {delta:>+14.2f}")

rouge_l_delta = comparison_results["metrics"]["rougeL"]["absolute_improvement"]
print("-" * 72)
if rouge_l_delta > 0:
    print(f"✅ Fine-tune cải thiện ROUGE-L {rouge_l_delta:+.2f} điểm trên tập test.")
elif rouge_l_delta < 0:
    print(f"⚠️ Fine-tune làm ROUGE-L giảm {rouge_l_delta:.2f} điểm trên tập test.")
else:
    print("ℹ️ ROUGE-L không thay đổi sau fine-tune.")

print("Lưu ý: ROUGE tăng là bằng chứng định lượng; vẫn nên đọc thủ công một số summary để kiểm tra tính đúng sự thật.")

In [ ]:
# Lưu metrics & config
train_metrics = train_result.metrics
train_metrics["train_runtime_formatted"] = format_duration(
    train_metrics.get("train_runtime", 0)
)

save_json(train_metrics, output_dir / "train_results.json")
save_json(eval_results, output_dir / "eval_results.json")
save_json(baseline_validation_results, output_dir / "baseline_validation_results.json")
save_json(baseline_test_results, output_dir / "baseline_test_results.json")
if test_results:
    save_json(test_results, output_dir / "test_results.json")
    save_json(comparison_results, output_dir / "before_after_comparison.json")
save_json(config_to_dict(config), output_dir / "resolved_config.json")

print("✅ Đã lưu tất cả metrics!")
print(f"   📁 {output_dir}")


---
## 9. Tự động huấn luyện phase 2

Phase 2 chỉ chạy khi checkpoint phase 1 có **validation ROUGE-L cao hơn model gốc**. Phase 2 luôn khởi tạo từ `outputs_phase_1/vit5_base/best`, không quay lại `VietAI/vit5-base`.

In [ ]:
PHASE_2_CONFIG_FILE = "configs/vit5_base_phase_2.yaml"

phase_2_completed = False
final_best_dir = best_dir
final_config = config
phase_2_summary = {
    "triggered": bool(phase_1_better_than_baseline),
    "gate_metric": "validation_rougeL",
    "phase_1_improvement": validation_rouge_l_delta,
    "phase_1_checkpoint": str(best_dir),
}

if not phase_1_better_than_baseline:
    print("⏭️ BỎ QUA PHASE 2")
    print(
        f"Phase 1 chưa vượt model gốc trên validation ROUGE-L "
        f"({validation_rouge_l_delta:+.2f})."
    )
    phase_2_summary["status"] = "skipped"
    save_json(phase_2_summary, output_dir / "phase_2_gate.json")
else:
    print("=" * 72)
    print("PHASE 1 TỐT HƠN BASELINE — BẮT ĐẦU HUẤN LUYỆN PHASE 2")
    print("=" * 72)

    phase_2_config = load_config(PHASE_2_CONFIG_FILE)
    phase_2_runtime_root = (
        Path("/kaggle/working")
        if is_kaggle
        else Path("/content") if Path("/content").exists() else Path.cwd()
    )
    phase_2_output_dir = phase_2_runtime_root / phase_2_config.training.output_dir
    phase_2_best_dir = phase_2_output_dir / "best"

    phase_2_config = apply_overrides(phase_2_config, {
        # Bắt buộc nối từ checkpoint tốt nhất phase 1.
        "model.name_or_path": str(best_dir),
        "training.output_dir": str(phase_2_output_dir),
        "training.precision": "bf16" if is_tpu else "fp16" if is_gpu else "auto",
        "training.optim": "adafactor" if (is_tpu or is_gpu) else "adamw_torch",
        "training.per_device_train_batch_size": 8 if is_tpu else 1,
        "training.per_device_eval_batch_size": 16 if is_tpu else 1,
        "training.gradient_accumulation_steps": 1 if is_tpu else 8 if is_gpu else 1,
        "training.gradient_checkpointing": bool(is_gpu),
        "training.label_smoothing_factor": 0.0 if is_gpu else phase_2_config.training.label_smoothing_factor,
        # GPU dùng ít beam hơn để tránh OOM trong model.generate.
        "generation.num_beams": phase_2_config.generation.num_beams if is_tpu else 2,
    })

    print(f"Nguồn model:  {phase_2_config.model.name_or_path}")
    print(f"Train data:   {phase_2_config.data.train_file}")
    print(f"Output:       {phase_2_output_dir}")
    print(f"Hardware:     {hardware}")
    print(f"Precision:    {phase_2_config.training.precision}")
    print(f"Epochs:       {phase_2_config.training.num_train_epochs}")
    print(f"Train batch:  {phase_2_config.training.per_device_train_batch_size}")
    print(f"Eval batch:   {phase_2_config.training.per_device_eval_batch_size}")
    print(f"Grad accum:   {phase_2_config.training.gradient_accumulation_steps}")
    print(f"Grad ckpt:    {phase_2_config.training.gradient_checkpointing}")
    print(f"Label smooth: {phase_2_config.training.label_smoothing_factor}")

    set_seed(phase_2_config.training.seed)
    phase_2_tokenizer = load_tokenizer(phase_2_config.model)
    phase_2_model = load_model(
        phase_2_config.model,
        phase_2_tokenizer,
        phase_2_config.generation,
    )
    if phase_2_config.training.gradient_checkpointing:
        enable_gradient_checkpointing(phase_2_model)
    if phase_2_config.training.freeze_encoder:
        freeze_encoder(phase_2_model)
    phase_2_model = apply_lora(phase_2_model, phase_2_config.lora)

    phase_2_datasets = load_and_preprocess(phase_2_tokenizer, phase_2_config.data)
    print(f"✅ Phase 2 train:      {len(phase_2_datasets['train'])}")
    print(f"✅ Phase 2 validation: {len(phase_2_datasets['validation'])}")
    print(f"✅ Phase 2 test:       {len(phase_2_datasets['test'])}")

    p2tc = phase_2_config.training
    phase_2_output_dir.mkdir(parents=True, exist_ok=True)
    phase_2_training_args = Seq2SeqTrainingArguments(
        output_dir=str(phase_2_output_dir),
        seed=p2tc.seed,
        num_train_epochs=p2tc.num_train_epochs,
        max_steps=p2tc.max_steps,
        per_device_train_batch_size=p2tc.per_device_train_batch_size,
        per_device_eval_batch_size=p2tc.per_device_eval_batch_size,
        gradient_accumulation_steps=p2tc.gradient_accumulation_steps,
        learning_rate=p2tc.learning_rate,
        weight_decay=p2tc.weight_decay,
        warmup_ratio=p2tc.warmup_ratio,
        lr_scheduler_type=p2tc.lr_scheduler_type,
        optim=p2tc.optim,
        label_smoothing_factor=p2tc.label_smoothing_factor,
        fp16=p2tc.precision == "fp16",
        bf16=p2tc.precision == "bf16",
        eval_strategy=p2tc.eval_strategy,
        eval_steps=p2tc.eval_steps,
        save_strategy=p2tc.save_strategy,
        save_steps=p2tc.save_steps,
        save_total_limit=p2tc.save_total_limit,
        logging_strategy="steps",
        logging_steps=p2tc.logging_steps,
        logging_first_step=True,
        disable_tqdm=True,
        metric_for_best_model=p2tc.metric_for_best_model,
        greater_is_better=p2tc.greater_is_better,
        load_best_model_at_end=p2tc.load_best_model_at_end,
        predict_with_generate=True,
        generation_max_length=phase_2_config.generation.max_length,
        report_to=["tensorboard"],
        logging_dir=str(phase_2_output_dir / "logs"),
        dataloader_drop_last=True,
    )

    phase_2_data_collator = DataCollatorForSeq2Seq(
        tokenizer=phase_2_tokenizer,
        model=phase_2_model,
        padding=True,
        label_pad_token_id=-100,
    )
    phase_2_compute_metrics = build_compute_metrics(phase_2_tokenizer)
    phase_2_progress_callback = KaggleProgressCallback(
        label="PHASE 2",
        log_every_steps=p2tc.logging_steps,
        heartbeat_seconds=60,
        log_file=phase_2_output_dir / "training_progress.log",
    )
    phase_2_callbacks = [phase_2_progress_callback]
    if p2tc.early_stopping_patience > 0:
        phase_2_callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=p2tc.early_stopping_patience,
            )
        )

    phase_2_trainer = Seq2SeqTrainer(
        model=phase_2_model,
        args=phase_2_training_args,
        train_dataset=phase_2_datasets["train"],
        eval_dataset=phase_2_datasets["validation"],
        tokenizer=phase_2_tokenizer,
        data_collator=phase_2_data_collator,
        compute_metrics=phase_2_compute_metrics,
        callbacks=phase_2_callbacks,
    )

    if is_gpu:
        torch.cuda.empty_cache()
    phase_2_start_time = time.time()
    try:
        phase_2_train_result = phase_2_trainer.train(
            resume_from_checkpoint=p2tc.resume_from_checkpoint,
        )
    finally:
        phase_2_progress_callback.stop()
    phase_2_elapsed = time.time() - phase_2_start_time
    print(f"✅ Phase 2 hoàn thành! Thời gian: {format_duration(phase_2_elapsed)}")

    phase_2_trainer.save_model(str(phase_2_best_dir))
    phase_2_tokenizer.save_pretrained(str(phase_2_best_dir))
    phase_2_eval_results = phase_2_trainer.evaluate(metric_key_prefix="eval")
    phase_2_test_results = phase_2_trainer.evaluate(
        eval_dataset=phase_2_datasets["test"],
        metric_key_prefix="test",
    )

    phase_2_train_metrics = phase_2_train_result.metrics
    phase_2_train_metrics["train_runtime_formatted"] = format_duration(
        phase_2_train_metrics.get("train_runtime", 0)
    )
    save_json(phase_2_train_metrics, phase_2_output_dir / "train_results.json")
    save_json(phase_2_eval_results, phase_2_output_dir / "eval_results.json")
    save_json(phase_2_test_results, phase_2_output_dir / "test_results.json")
    save_json(config_to_dict(phase_2_config), phase_2_output_dir / "resolved_config.json")

    phase_2_completed = True
    final_best_dir = phase_2_best_dir
    final_config = phase_2_config
    phase_2_summary.update({
        "status": "completed",
        "phase_2_checkpoint": str(phase_2_best_dir),
        "phase_2_eval_rougeL": phase_2_eval_results.get("eval_rougeL"),
        "phase_2_test_rougeL": phase_2_test_results.get("test_rougeL"),
    })
    save_json(phase_2_summary, output_dir / "phase_2_gate.json")

    print("\n" + "=" * 72)
    print("KẾT QUẢ PHASE 2")
    print("=" * 72)
    print(f"  Best checkpoint: {phase_2_best_dir}")
    print(f"  Validation ROUGE-L: {phase_2_eval_results.get('eval_rougeL')}")
    print(f"  Test ROUGE-L:       {phase_2_test_results.get('test_rougeL')}")

    del phase_2_trainer
    del phase_2_model
    del phase_2_data_collator
    gc.collect()

---
## 10. Test thử model

In [ ]:
from src.predict import summarize

test_text = """Thủ tướng Chính phủ vừa phê duyệt đề án phát triển ứng dụng 
trí tuệ nhân tạo tại Việt Nam giai đoạn 2025-2030. Theo đó, Việt Nam đặt 
mục tiêu trở thành một trong những trung tâm đổi mới sáng tạo về AI trong 
khu vực ASEAN. Đề án tập trung vào 5 lĩnh vực ưu tiên gồm y tế, giáo dục, 
nông nghiệp, giao thông và sản xuất công nghiệp."""

summary = summarize(
    text=test_text,
    model_path=str(final_best_dir),
    config=final_config,
)

print(f"🔎 Checkpoint inference: {final_best_dir}")
print(f"   Phase 2 completed: {phase_2_completed}")
print("📄 Bài gốc:")
print(test_text.strip())
print("\n📝 Tóm tắt:")
print(summary)